In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e4/train.csv
/kaggle/input/competitions/playground-series-s6e4/test.csv


# 1. Data Loading

In [2]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler

# It looks like the input directory is '/kaggle/input/playground-series-s6e4', let's use that
data_path = '/kaggle/input/competitions/playground-series-s6e4/'

try:
    train_df = pd.read_csv(data_path + 'train.csv')
    test_df = pd.read_csv(data_path + 'test.csv')
    sample_submission_df = pd.read_csv(data_path + 'sample_submission.csv')
except FileNotFoundError as e:
    print(f"Error: {e}")
    print("Trying the local 'Data' directory...")
    train_df = pd.read_csv('Data/train.csv')
    test_df = pd.read_csv('Data/test.csv')
    sample_submission_df = pd.read_csv('Data/sample_submission.csv')

print("Train data shape:", train_df.shape)
print("Test data shape:", test_df.shape)

Train data shape: (630000, 21)
Test data shape: (270000, 20)


# 2. Data Preprocessing

In [3]:
X = train_df.drop('Irrigation_Need', axis=1)
y = train_df['Irrigation_Need']

categorical_features = X.select_dtypes(include=['object']).columns
numerical_features = X.select_dtypes(include=['number']).columns

for col in categorical_features:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    test_df[col] = le.transform(test_df[col])

scaler = StandardScaler()
numerical_features_no_id = [col for col in numerical_features if col != 'id']
X[numerical_features_no_id] = scaler.fit_transform(X[numerical_features_no_id])
test_df[numerical_features_no_id] = scaler.transform(test_df[numerical_features_no_id])

# 3. Model Training

In [4]:
model = lgb.LGBMClassifier(random_state=42)
model.fit(X, y)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.018862 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2960
[LightGBM] [Info] Number of data points in the train set: 630000, number of used features: 20
[LightGBM] [Info] Start training from score -3.400769
[LightGBM] [Info] Start training from score -0.532441
[LightGBM] [Info] Start training from score -0.968947


LGBMClassifier(random_state=42)

# 4. Prediction and Submission

In [5]:
test_predictions = model.predict(test_df)

submission_df = pd.DataFrame({'id': test_df['id'], 'Irrigation_Need': test_predictions})
submission_df.to_csv('submission.csv', index=False)

print("Submission file created successfully!")
print(submission_df.head())

Submission file created successfully!
       id Irrigation_Need
0  630000             Low
1  630001             Low
2  630002             Low
3  630003             Low
4  630004             Low
